# 🏆 XAUUSD EA — 17-Model Ensemble Training
### Fusion Markets M1 Gold Scalper

---

**Before running anything:**
1. Click `Runtime` → `Change runtime type` → Select **T4 GPU** → Save
2. Run cells **in order**, top to bottom
3. If Colab disconnects: reconnect and re-run from Cell 4 — already-trained models are automatically skipped

---

| Model | Architecture | Status |
|---|---|---|
| Transformer | Global self-attention | Trained from scratch |
| LSTM | BiLSTM + attention | Trained from scratch |
| TCN | Dilated conv | Trained from scratch |
| PatchTST | Patch-based SOTA 2023 | Trained from scratch |
| TFT | Financial VSN+GRN | Trained from scratch |
| N-HiTS | Hierarchical macro→micro | Trained from scratch |
| iTransformer | Feature-space attention 2024 | Trained from scratch |
| Mamba | Selective state space 2023 | Trained from scratch |
| DLinear | Trend decomposition | Trained from scratch |
| xLSTM | Matrix memory LSTM 2024 | Trained from scratch |
| TimesNet | 2D temporal via FFT 2023 | Trained from scratch |
| **Chronos** | **Amazon pre-trained T5** | **Encoder FROZEN — head only** |
| TimeMixer | Multi-scale mixing ICLR 2024 | Trained from scratch |
| SOFTS | Star aggregate NeurIPS 2024 | Trained from scratch |
| GradBoost | sklearn HistGradBoost | Tree-based |
| LightGBM | Fast boosting | Tree-based |
| CatBoost | Ordered boosting | Tree-based |

## ✅ Cell 1 — Check GPU
Make sure you have a T4 GPU before installing anything.

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'✅ GPU detected: {gpu}  ({vram:.1f} GB VRAM)')
else:
    print('❌ No GPU detected!')
    print('Go to: Runtime → Change runtime type → T4 GPU → Save')
    print('Then: Runtime → Restart session, and re-run from Cell 1')

## 📦 Cell 2 — Install Dependencies
Takes ~3 minutes. Run once per session.

In [ ]:
print('Installing dependencies...')

# Core ML stack
!pip install -q scikit-learn==1.9.0
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Gradient boosting trio
!pip install -q lightgbm catboost

# Chronos foundation model (Amazon pre-trained)
!pip install -q git+https://github.com/amazon-science/chronos-forecasting.git

# Data + utilities
!pip install -q pandas numpy yfinance ta tqdm

print('\n✅ All dependencies installed!')

# Verify key versions
import sklearn, lightgbm, catboost, torch
print(f'  scikit-learn : {sklearn.__version__}  (needs 1.9.0)')
print(f'  lightgbm     : {lightgbm.__version__}')
print(f'  catboost     : {catboost.__version__}')
print(f'  torch        : {torch.__version__}')
print(f'  CUDA         : {torch.version.cuda}')

## 💾 Cell 3 — Mount Google Drive
Checkpoints are saved here. If Colab disconnects, your progress is safe.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/claude_ea/checkpoints'
PROGRESS_DIR   = '/content/drive/MyDrive/claude_ea/inprogress'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(PROGRESS_DIR,   exist_ok=True)

print(f'✅ Drive mounted')
print(f'   Checkpoints → {CHECKPOINT_DIR}')

# Show any existing checkpoints
existing = [f for f in os.listdir(CHECKPOINT_DIR) if f.endswith('.pth') or f.endswith('.joblib')]
if existing:
    print(f'\n📂 Existing checkpoints found ({len(existing)}):')
    for f in sorted(existing):
        size = os.path.getsize(os.path.join(CHECKPOINT_DIR, f)) / 1024**2
        print(f'   ✓  {f}  ({size:.1f} MB)')
else:
    print('\n📂 No existing checkpoints — fresh training run')

## 📥 Cell 4 — Clone the EA Repository

In [ ]:
import os

if os.path.exists('/content/Claude'):
    print('Repository already cloned — pulling latest...')
    !cd /content/Claude && git pull origin feature/5-model-ensemble-tcn-lgbm
else:
    print('Cloning repository...')
    !git clone --branch feature/5-model-ensemble-tcn-lgbm \
        https://github.com/gagandocx/Claude.git /content/Claude

os.chdir('/content/Claude/python_bridge')
print(f'\n✅ Working directory: {os.getcwd()}')
print('\nModel files:')
!ls models/*.py | head -20

## 📊 Cell 5 — Load Your Fusion Markets CSV

**Option A (recommended):** Auto-download from your GitHub uploads repo

**Option B:** Upload manually from your PC

Run the option that applies to you.

In [ ]:
# ─── OPTION A: Auto-download from gagandocx/Uploads ───
import urllib.request, os

CSV_URL  = 'https://raw.githubusercontent.com/gagandocx/Uploads/main/bars_export_XAUUSD.csv'
CSV_PATH = '/content/bars_export_XAUUSD.csv'

if not os.path.exists(CSV_PATH):
    print('Downloading bars_export_XAUUSD.csv...')
    urllib.request.urlretrieve(CSV_URL, CSV_PATH)

import pandas as pd
df = pd.read_csv(CSV_PATH, sep=None, engine='python', nrows=5)
size = os.path.getsize(CSV_PATH) / 1024**2
print(f'✅ CSV ready  ({size:.1f} MB)')
print(f'   Columns: {list(df.columns)}')
print(df.head(3).to_string())

In [ ]:
# ─── OPTION B: Upload from your PC ───
# Only run this if Option A didn't work

from google.colab import files
print('Select bars_export_XAUUSD.csv from your PC...')
uploaded = files.upload()
fname = list(uploaded.keys())[0]
import shutil
shutil.move(fname, '/content/bars_export_XAUUSD.csv')
print(f'✅ Uploaded: {fname}')

## 🔧 Cell 6 — Configure Training Settings
Edit these before running Cell 7.

In [ ]:
# ─── TRAINING CONFIGURATION ───────────────────────────────────────────────────

import os

# Where the trained checkpoints will be saved
CHECKPOINT_DIR = '/content/drive/MyDrive/claude_ea/checkpoints'
PROGRESS_DIR   = '/content/drive/MyDrive/claude_ea/inprogress'

# CSV path
CSV_PATH = '/content/bars_export_XAUUSD.csv'

# Save in-progress checkpoint every N epochs (protects against disconnects)
SAVE_EVERY = 5

# Patch the train_colab.py config with our settings
config_patch = f"""
import sys
sys.path.insert(0, '/content/Claude/python_bridge')
import train_colab as _tc
_tc.CSV_PATH       = '{CSV_PATH}'
_tc.CHECKPOINT_DIR = '{CHECKPOINT_DIR}'
_tc.PROGRESS_DIR   = '{PROGRESS_DIR}'
_tc.SAVE_EVERY     = {SAVE_EVERY}
_tc.USE_GITHUB_CSV = False   # we already have the CSV
"""

with open('/tmp/patch_config.py', 'w') as f:
    f.write(config_patch)

print('✅ Training config:')
print(f'   CSV          : {CSV_PATH}')
print(f'   Checkpoints  : {CHECKPOINT_DIR}')
print(f'   Save every   : {SAVE_EVERY} epochs')
print(f'   Resume       : Yes (auto-skips trained models)')

## 🚀 Cell 7 — Train All 17 Models

**Expected time on T4 GPU: 2–4 hours**

Progress bars show live loss/accuracy for each model.

If Colab disconnects → reconnect → re-run Cells 3 and 4 → re-run this cell.
Already-finished models will show `✓ ALREADY TRAINED` and be skipped.

In [ ]:
import os, sys
os.chdir('/content/Claude/python_bridge')
sys.path.insert(0, '/content/Claude/python_bridge')

# Apply config patch
exec(open('/tmp/patch_config.py').read())

# Run training
from train_colab import train_all_colab
train_all_colab()

## 📈 Cell 8 — Verify Results
Run after training completes to confirm models beat the random baseline.

In [ ]:
import os

CHECKPOINT_DIR = '/content/drive/MyDrive/claude_ea/checkpoints'

print('=== CHECKPOINT SUMMARY ===')
print()

neural_files = [
    ('transformer.pth',  'Transformer'),
    ('lstm.pth',         'LSTM'),
    ('tcn.pth',          'TCN'),
    ('patch_tst.pth',    'PatchTST'),
    ('tft.pth',          'TFT'),
    ('nhits.pth',        'N-HiTS'),
    ('itransformer.pth', 'iTransformer'),
    ('mamba.pth',        'Mamba'),
    ('dlinear.pth',      'DLinear'),
    ('xlstm.pth',        'xLSTM'),
    ('timesnet.pth',     'TimesNet'),
    ('chronos.pth',      'Chronos'),
    ('timemixer.pth',    'TimeMixer'),
    ('softs.pth',        'SOFTS'),
    ('gradient_boost.joblib', 'GradBoost'),
    ('xgboost_extra.joblib',  'LightGBM/XGB'),
    ('catboost.joblib',       'CatBoost'),
    ('meta_learner.joblib',   'Meta-Learner'),
]

total_size = 0
done = 0
for fname, label in neural_files:
    fpath = os.path.join(CHECKPOINT_DIR, fname)
    if os.path.exists(fpath):
        size = os.path.getsize(fpath) / 1024**2
        total_size += size
        done += 1
        print(f'  ✅ {label:<20} {fname:<30} {size:6.1f} MB')
    else:
        print(f'  ❌ {label:<20} {fname:<30} MISSING')

print()
print(f'  Models saved : {done}/{len(neural_files)}')
print(f'  Total size   : {total_size:.1f} MB')
print(f'  Location     : {CHECKPOINT_DIR}')

## 💻 Cell 9 — Download Checkpoints to Your PC

**Option A (Google Drive — recommended):**
The checkpoints are already in your Google Drive at:
`My Drive/claude_ea/checkpoints/`

Download the `checkpoints` folder and copy it to your trading PC:
```
C:\Users\gagan\...\python_bridge\checkpoints\
```

**Option B: Download ZIP directly from Colab**

In [ ]:
# Create a zip of all checkpoints for direct download
import shutil, os

CHECKPOINT_DIR = '/content/drive/MyDrive/claude_ea/checkpoints'
ZIP_PATH       = '/content/ea_checkpoints.zip'

print('Creating checkpoint zip...')
shutil.make_archive('/content/ea_checkpoints', 'zip', CHECKPOINT_DIR)

size = os.path.getsize(ZIP_PATH) / 1024**2
print(f'✅ Zip created: {ZIP_PATH}  ({size:.1f} MB)')

from google.colab import files
print('Starting download...')
files.download(ZIP_PATH)

## 🔁 Cell 10 — Install on Trading PC

After downloading `ea_checkpoints.zip`:

1. Extract the zip
2. Copy all `.pth` and `.joblib` files to:
   ```
   C:\Users\gagan\...\python_bridge\checkpoints\
   ```
3. Start the EA as normal:
   ```bash
   python main.py --live
   ```

The system will automatically load all 17 trained models. You will see in the logs:
```
[Ensemble] Loaded transformer.pth
[Ensemble] Loaded lstm.pth
...
[Ensemble] Load complete. gb=True xgb=True cb=True meta=True
```

---

**Support:** If any model fails to load, it automatically falls back to weighted average — the EA continues running safely.